# Quickstart: Research to Trading Pipeline
This notebook walks through data ingestion, feature engineering, agent training, and evaluation.

In [ ]:
from e2e_bcrt_re_to_td import DEFAULT_CONFIG
from e2e_bcrt_re_to_td.data.ingestion import run_data_ingestion
from e2e_bcrt_re_to_td.features import assemble_technical_features, SentimentExtractor
from e2e_bcrt_re_to_td.agents import PPOTradingAgent, ResearchAssistant
from e2e_bcrt_re_to_td.evaluation import simple_backtest

config = DEFAULT_CONFIG
config.ensure_directories()

In [ ]:
result = run_data_ingestion(
    symbols=config.data.symbols,
    start_date=config.data.start_date,
    end_date=config.data.end_date,
    interval=config.data.interval,
)
symbol = config.data.symbols[0]
market_df = result.market_data[symbol]
market_df.tail()

In [ ]:
technical = assemble_technical_features(market_df, config.features.technical_indicators)
technical.tail()

In [ ]:
agent = PPOTradingAgent(config=config.agent.__dict__, checkpoint_dir=config.agent.checkpoint_dir)
agent.train(market_df, technical)
signals = technical.apply(lambda row: agent.act(row.to_dict())["allocation"], axis=1)
backtest = simple_backtest(prices=market_df["Close"], signals=signals)
backtest.metrics

In [ ]:
sentiment_extractor = SentimentExtractor(model=config.features.sentiment_model)
assistant = ResearchAssistant(sentiment_extractor)
context = assistant.build_context(market_df.tail(5), technical.tail(5))
documents = [
    "Positive revenue surprise lifts outlook.",
    "Macro headwinds could weigh on margins.",
]
report = assistant.generate_report(documents, agent_signals=backtest.metrics, context=context)
report